# SantéNLP: fine-tune Bio_ClinicalBERT on Colab

Trains the specialty classifier (`src/classify.py`) on a GPU and saves the results to Google Drive.

**Before you start:** *Runtime → Change runtime type → T4 GPU*. Then run the cells from top to bottom.

What gets trained, for each of the 2 ways of shortening long reports (`head`, `head_tail`):
a main model on the saved split + 5 seeds (new train/val split each, same test set) = 12 fine-tunings in total.

Results go to `MyDrive/SanteNLP/`:
- `outputs/runs/<date>_bert_specialty_<strategy>/`: config, metrics, test predictions (row numbers only)
- `models/bert_specialty/<strategy>/`: the main model of each strategy (~420 MB each)

### Avoid disconnects
Free Colab disconnects a runtime that looks idle (roughly 90 minutes without interaction) and stops any runtime after
about 12 hours. During training:
- keep this tab **open and visible** (not minimized, not in a background tab) and keep the computer awake;
- come back every 30–60 minutes and click or scroll in the notebook.

### After a disconnect or restart
Every cell can be rerun on its own: each one re-enters the code folder and rebuilds the Drive paths it needs,
and the training cells stop with a clear error if Drive is not mounted.
- **Runtime restarted** (*Runtime → Restart session*, files still there): rerun step 2, then the step you need.
- **New machine** (*Runtime disconnected*, `/content/SanteNLP` is gone): rerun steps 2 to 5, then the step you need.

A strategy's results are written to Drive only when all its 6 fine-tunings are done, so an interrupted
training step starts over; finished steps are already on Drive and do not need to be rerun.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys
print("Python", sys.version)

## 2. Connect Google Drive

A window asks for permission: accept it. Results are written to Drive so they survive the end of the Colab session.
Rerun this cell after any restart or disconnect (if Drive is already connected, it does nothing).

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/SanteNLP", exist_ok=True)
!ls -la /content/drive/MyDrive/SanteNLP

## 3. Get the code

In [ ]:
REPO = "https://github.com/mohcine00errachind/SanteNLP.git"
!rm -rf /content/SanteNLP && git clone --quiet --depth 1 {REPO} /content/SanteNLP
%cd /content/SanteNLP
!git log -1 --format="Code version: %h  %s"

## 4. Install the pinned libraries

Colab already has a GPU build of PyTorch, so `torch` is **not** reinstalled. Also skipped: the spaCy word vectors
(NER only), notebook tools (Colab has its own) and Streamlit (app only).

pip may print *dependency conflict* warnings about Colab's own pre-installed packages: they are expected and
harmless here, because everything that uses these libraries runs in a separate Python process (`!python ...`).
**No restart is needed** after this step (a restart would only be needed to import the new numpy/pandas inside
the notebook itself, which this notebook never does).

The last cell of this step checks, in a new Python process, that torch is still Colab's own build and that `transformers` can load its `Trainer`. If it fails, do not continue: *Runtime → Disconnect and delete runtime*, then start again from step 1.

In [ ]:
import os
os.chdir("/content/SanteNLP")  # fails here if step 3 has not run on this machine
# Colab's own torch version, recorded once per machine (before anything is installed)
!test -f /tmp/colab_torch_version.txt || python -c "import torch; print(torch.__version__)" > /tmp/colab_torch_version.txt
# Keep every requirement except these (a package name followed by ==, @, a space or the end of the line)
!grep -v -E "^(torch|en_core_web_md|ipykernel|nbformat|nbclient|streamlit)([=@ ]|$)" requirements.txt > /tmp/requirements_colab.txt
!cat /tmp/requirements_colab.txt
!pip install -q -r /tmp/requirements_colab.txt

In [ ]:
%%bash
# New Python process: sees exactly what training (step 7) will see
python - <<'EOF'
import torch
expected = open("/tmp/colab_torch_version.txt").read().strip()
assert torch.__version__ == expected, (
    f"torch was changed by pip: {torch.__version__} instead of Colab's {expected}. "
    "Disconnect and delete the runtime, then start again from step 1.")
from transformers import Trainer  # fails if torchvision/torch do not match
import numpy, transformers
print("OK | torch", torch.__version__, "(Colab's) | CUDA:", torch.cuda.is_available(),
      "| transformers", transformers.__version__, "| numpy", numpy.__version__)
EOF

## 5. Download and prepare the data

Same script as locally. The check below makes sure Colab built **exactly** the same train/val/test split.
It runs in a new Python process, so it uses the numpy/pandas just installed (no restart needed).

In [ ]:
import os
os.chdir("/content/SanteNLP")
!python -m src.data --config configs/data.yaml

In [ ]:
%%bash
cd /content/SanteNLP
python - <<'EOF'
from src.data import fingerprint, load_mtsamples_splits

# Fingerprints of the split built locally (Windows), see src/data.py
EXPECTED = {"train": 5892761080162288042, "val": 6317211258887222346, "test": 1618043657115009311}
splits = load_mtsamples_splits()
for name, df in splits.items():
    assert fingerprint(df) == EXPECTED[name], f"{name} split differs from the local one!"
print("Same split as locally:", {name: len(df) for name, df in splits.items()})
EOF

## 6. Quick check (about 1 minute)

A tiny run (24 reports, 1 epoch) to catch errors before the long training. Its scores mean nothing.

In [ ]:
import os
os.chdir("/content/SanteNLP")
!python -m src.classify --smoke --runs-dir /tmp/smoke

## 7. Train: `head` strategy (first 510 tokens)

6 fine-tunings (main model + 5 seeds). Each line printed is one finished model.
At the end, the cell lists the files saved on Drive: check that the run folder and the model are there.

In [ ]:
import os
os.chdir("/content/SanteNLP")
from src.colab import drive_dirs, show_files

RUNS, MODELS = drive_dirs()  # stops with a clear error if Drive is not mounted
!python -m src.classify --strategies head --runs-dir "{RUNS}" --model-dir "{MODELS}"
show_files(RUNS)
show_files(MODELS / "head")

## 8. Train: `head_tail` strategy (first 128 + last 382 tokens)

In [ ]:
import os
os.chdir("/content/SanteNLP")
from src.colab import drive_dirs, show_files

RUNS, MODELS = drive_dirs()
!python -m src.classify --strategies head_tail --runs-dir "{RUNS}" --model-dir "{MODELS}"
show_files(RUNS)
show_files(MODELS / "head_tail")

## 9. Summary

In [ ]:
import json, os
os.chdir("/content/SanteNLP")
from src.colab import drive_dirs

RUNS, MODELS = drive_dirs()
paths = sorted(RUNS.glob("*_bert_specialty_*"))
assert paths, f"No BERT runs found in {RUNS}"
print(f"{'run':45s} {'main test F1':>12s} {'seeds: mean ± std':>20s}")
for path in paths:
    m = json.loads((path / "metrics.json").read_text(encoding="utf-8"))
    print(f"{path.name:45s} {m['main']['test']['macro_f1']:12.3f} "
          f"{m['test_macro_f1_mean']:13.3f} ± {m['test_macro_f1_std']:.3f}")
print("\nBaseline (TF-IDF + logistic regression): main 0.954, seeds 0.940 ± 0.003")

## 10. Next step

Copy these two Drive folders into the project on your computer (see the README):
- `MyDrive/SanteNLP/outputs/runs/*_bert_specialty_*` → `outputs/runs/`
- `MyDrive/SanteNLP/models/bert_specialty/` → `models/bert_specialty/`